In [ ]:
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
  subprocess.run(
    ["git", "clone", "--recursive", "https://github.com/yangyi02/droid.git", "/content/droid"],
    check=True,
  )
  os.chdir("/content/droid")

REPO_DIR = os.getcwd()
CACHE_DIR = os.path.join(REPO_DIR, "data", "cache")
GCS_OUTPUT = "gs://dm-tapnet/tmp/droid"

if IN_COLAB:
  from google.colab import auth

  auth.authenticate_user()

  subprocess.run([sys.executable, "-m", "pip", "install", "-U", "ipython"], check=True)

  subprocess.run(["bash", "setup.sh", "--no-depth"], check=True)

In [ ]:
%reload_ext autoreload
%autoreload 2

import random

import cv2
import matplotlib.colors
import mediapy as media
import numpy as np

from config import get_config
import core.geometry
import core.io
import core.physics
import core.visualization
import compute_tracks

config = get_config()

In [ ]:
GREEN, RED, BLUE, AMBER = "#2da44e", "#cf222e", "#0969da", "#bf8700"
RNG = np.random.default_rng(0)

SUBSAMPLE = 30000  # candidates kept per stage; None keeps every one, like the pipeline
MAX_FRAMES = 60  # frames per video; None renders the whole episode
TGT_SIZE = (288, 512)  # one camera panel
N_SHOW = 9000  # points per row, shared across colours so their densities stay comparable
RADIUS = 1  # dot radius in pixels


def rgb255(color):
  return np.array(matplotlib.colors.to_rgb(color)) * 255


def pick_candidates(n_points, keep=SUBSAMPLE):
  if keep is None or keep >= n_points:
    return np.arange(n_points)
  return np.sort(RNG.choice(n_points, keep, replace=False))


def paint_frames(view, uv, vis, sel, vis_rgb, hidden_rgb, tgt_size, max_frames, radius):
  """One filled dot per point per frame, no trace: vis_rgb where vis is true, hidden_rgb where it is false."""
  video = np.asarray(episode["camera"][cam_ids[view]]["video_rgb"])[:max_frames]
  src_height, src_width = video.shape[1:3]
  height, width = tgt_size
  frames = np.ascontiguousarray(media.resize_video(video, tgt_size))

  pts = np.round(uv[view, :max_frames][:, sel] * [width / src_width, height / src_height]).astype(np.int32)
  colors = np.where(vis[view, :max_frames][:, sel, None], vis_rgb, hidden_rgb).astype(np.uint8)
  offsets = [
    (dx, dy) for dy in range(-radius, radius + 1) for dx in range(-radius, radius + 1) if dx * dx + dy * dy <= radius * radius
  ]

  for t in range(len(frames)):
    for dx, dy in offsets:
      x, y = pts[t, :, 0] + dx, pts[t, :, 1] + dy
      inside = (x >= 0) & (x < width) & (y >= 0) & (y < height)
      frames[t, y[inside], x[inside]] = colors[t][inside]

  return frames


def multiview_frames(uv, vis, sel, vis_rgb, hidden_rgb, tgt_size, max_frames, radius, banner=None, legend=()):
  """Every camera side by side. sel and the colours may be one array, or a list with one entry per camera."""
  panels = []
  for view, cam_id in enumerate(cam_ids):
    args = (x[view] if isinstance(x, list) else x for x in (sel, vis_rgb, hidden_rgb))
    frames = paint_frames(view, uv, vis, *args, tgt_size, max_frames, radius)
    for frame in frames:
      core.visualization.draw_label(frame, f"Cam [{cam_id[:8]}]", (8, 20), 0.5, (255, 255, 255), 1, 3)
    panels.append(frames)

  block = np.concatenate(panels, axis=2)
  for frame in block:
    if banner:
      core.visualization.draw_label(frame, banner, (8, 46), 0.6, (255, 255, 255), 1, 3)
    x = 8
    for text, color in legend:
      core.visualization.draw_label(frame, text, (x, 72 if banner else 46), 0.55, tuple(int(c) for c in color), 1, 3)
      x += cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 0.55, 1)[0][0] + 28

  return block


def query_video(uv, gap, query_view, tol, title, n_show=4000, max_frames=MAX_FRAMES, tgt_size=(216, 384), radius=RADIUS):
  """One row per query view: the candidates that camera seeded, seen from every camera. green = vis, red = occluded."""
  vis = gap >= -tol
  legend = [("green = vis", rgb255(GREEN)), ("red = occluded", rgb255(RED))]

  rows = []
  for view, cam_id in enumerate(cam_ids):
    own = np.flatnonzero(query_view == view)
    sel = np.sort(RNG.choice(own, n_show, replace=False) if len(own) > n_show else own)
    vis_rgb = np.tile(rgb255(GREEN), (len(sel), 1))
    hidden_rgb = np.tile(rgb255(RED), (len(sel), 1))
    banner = f"query view {view} [{cam_id[:8]}] - {len(own):,} candidates, depth_tolerance = {tol:g} m"
    rows.append(multiview_frames(uv, vis, sel, vis_rgb, hidden_rgb, tgt_size, max_frames, radius, banner, legend))

  media.show_video(np.concatenate(rows, axis=1), fps=10, title=title)


def flag_video(uv, vis, rows, title, n_show=N_SHOW, max_frames=MAX_FRAMES, tgt_size=TGT_SIZE, radius=RADIUS, dim=0.3):
  """rows: [(banner, [(label, mask, colour)])]. Colour is the fate, brightness is vis on that frame."""
  blocks = []
  for banner, layers in rows:
    idx = np.flatnonzero(np.logical_or.reduce([mask for _, mask, _ in layers]))
    sel = np.sort(RNG.choice(idx, n_show, replace=False) if len(idx) > n_show else idx)

    vis_rgb = np.zeros((len(sel), 3))
    for _, mask, color in layers:
      vis_rgb[mask[sel]] = rgb255(color)

    legend = [(f"{label}  {int(mask.sum()):,}", rgb255(color)) for label, mask, color in layers]
    blocks.append(multiview_frames(uv, vis, sel, vis_rgb, vis_rgb * dim, tgt_size, max_frames, radius, banner, legend))

  media.show_video(np.concatenate(blocks, axis=1), fps=10, title=title)

In [ ]:
EPISODE_ID = None  # None -> random successful episode

serials_db, id_to_path, extrinsics_db, _ = core.io.load_metadata(config)

with open(os.path.join(REPO_DIR, "episodes_success.txt")) as f:
  valid_ids = sorted(line.strip() for line in f if line.strip())

episode_id = EPISODE_ID or random.choice(valid_ids)

depth_root = os.path.join(CACHE_DIR, "depth")
ext_root = os.path.join(CACHE_DIR, "extrinsics")
for stage, root in (("depth", depth_root), ("extrinsics", ext_root)):
  dst = os.path.join(root, episode_id)
  if not os.path.exists(dst):
    subprocess.run(["gcloud", "storage", "rsync", "-r", f"{GCS_OUTPUT}/{stage}/{episode_id}", dst], check=True)

episode = core.io.load_depth_data(episode_id, depth_root, load_video=True)
poses = core.io.load_extrinsics(episode, ext_root)
pb_renderer = core.physics.PyBulletRenderer(config.paths.urdf, gpu=config.render.gpu)

cam_ids = list(episode["camera"])
n_frames = len(episode["robot"]["joint_positions"])
print(f"{episode_id} | {n_frames} frames | cameras {cam_ids} | wrist {episode['meta']['wrist_serial']}")

In [ ]:
# the urdf drawn through each camera's extrinsics: if the blue mask does not sit on the arm, the pose is wrong
seg_frames = core.visualization.render_segmentation_video(episode, poses, pb_renderer, max_frames=MAX_FRAMES)
media.show_video(seg_frames, fps=10, title="Robot Mask Overlay")


In [ ]:
static_xyz_full, static_view_full = compute_tracks.find_static_candidates(
  episode, poses, pb_renderer, config.tracks.match_radius, config.tracks.mask_margin
)

static_idx = pick_candidates(len(static_view_full))
static_xyz, static_view = static_xyz_full[static_idx], static_view_full[static_idx]

print(f"static candidates: {len(static_view_full):,} -> {len(static_view):,} after subsample")


In [ ]:
def project_static_debug(static_points_3d, episode, poses, pb_renderer):
  """compute_tracks.project_static_tracks:135-163 without line 160 — both gap channels kept, no vis."""
  robot = episode["robot"]
  n_frames = len(robot["joint_positions"])
  n_views, n_points = len(episode["camera"]), len(static_points_3d)

  uv = np.zeros((n_views, n_frames, n_points, 2), dtype=np.float32)
  urdf_gap = np.zeros((n_views, n_frames, n_points), dtype=np.float32)
  sensor_gap = np.zeros((n_views, n_frames, n_points), dtype=np.float32)

  for t in range(n_frames):
    pb_renderer.update_robot_pose(robot["joint_positions"][t], gripper_state=robot["gripper_positions"][t])

    for view, (cam_id, cam_data) in enumerate(episode["camera"].items()):
      K = cam_data["K"]
      height, width = cam_data["raw_depth"][t].shape
      T_cam2world = poses[cam_id]["extrinsics"][t]
      urdf_depth = pb_renderer.render_depth(T_cam2world, K, width, height)

      u, v, z_pred = core.geometry.project_points(static_points_3d, K, T_cam2world)
      uv[view, t] = np.stack([u, v], axis=1)

      z_urdf = core.geometry.sample_depth(urdf_depth, u, v, z_pred)
      z_sensor = core.geometry.sample_depth(cam_data["raw_depth"][t], u, v, z_pred)
      measured = np.stack([z_urdf, z_sensor])
      urdf_gap[view, t], sensor_gap[view, t] = np.where(measured == 0, np.inf, measured) - z_pred

  return uv, urdf_gap, sensor_gap


static_uv, static_urdf_gap, static_sensor_gap = project_static_debug(static_xyz, episode, poses, pb_renderer)

# line 160: the arm occludes and the scene occludes, either one hides the point
static_min_gap = np.minimum(static_urdf_gap, static_sensor_gap)


In [ ]:
STATIC_DEPTH_TOL = config.tracks.static_depth_tolerance

static_vis = static_min_gap >= -STATIC_DEPTH_TOL
static_gap = static_sensor_gap  # line 161: gone only ever asks the sensor, the urdf cannot see through anything

query_video(static_uv, static_min_gap, static_view, STATIC_DEPTH_TOL, "Static vis — one row per query view")


In [ ]:
STATIC_MIN_RUN_FRACTION = config.tracks.min_run_fraction
STATIC_FLICKER = config.tracks.flicker

# compute_tracks.filter_static_tracks:166-178, with the per-view .any(axis=0) folded in:
# one camera flagging a point drops it from every camera.


def gone_flag(min_run_fraction):
  """Line 173: min_frames CONSECUTIVE frames of seeing past the point, and only if it was visible at t=0."""
  min_frames = int(min_run_fraction * n_frames)
  seen_through = np.isfinite(static_gap) & (static_gap > STATIC_DEPTH_TOL)
  windows = np.lib.stride_tricks.sliding_window_view(seen_through, min_frames, axis=1)
  return (windows.all(axis=-1).any(axis=1) & static_vis[:, 0]).any(axis=0)


def jitter_flag(flicker):
  """Line 174: the fraction of frame transitions where vis flips."""
  return ((static_vis[:, 1:] != static_vis[:, :-1]).mean(axis=1) > flicker).any(axis=0)


static_gone = gone_flag(STATIC_MIN_RUN_FRACTION)
static_jitters = jitter_flag(STATIC_FLICKER)
static_keep = ~(static_gone | static_jitters)

flag_video(
  static_uv,
  static_vis,
  [(
    f"tol={STATIC_DEPTH_TOL:g}  min_run_fraction={STATIC_MIN_RUN_FRACTION:g} "
    f"({int(STATIC_MIN_RUN_FRACTION * n_frames)} frames)  flicker={STATIC_FLICKER:g}",
    [
      ("kept", static_keep, GREEN),
      ("gone", static_gone, RED),
      ("jitters (only)", static_jitters & ~static_gone, AMBER),
    ],
  )],
  "Static candidates by fate",
)


In [ ]:
GONE_GRID = [0.05, 0.10, 0.20, 0.40]

rows = []
for fraction in GONE_GRID:
  flagged = gone_flag(fraction)
  rows.append((
    f"min_run_fraction = {fraction:g}  ({int(fraction * n_frames)} of {n_frames} frames)  gone {int(flagged.sum()):,}",
    [("kept", ~flagged, GREEN), ("gone", flagged, RED)],
  ))

flag_video(static_uv, static_vis, rows, "Static gone — one row per min_run_fraction")


In [ ]:
FLICKER_GRID = [0.02, 0.05, 0.10, 0.20]

rows = []
for flicker in FLICKER_GRID:
  flagged = jitter_flag(flicker)
  rows.append((
    f"flicker = {flicker:g}  ({flicker * (n_frames - 1):.0f} of {n_frames - 1} transitions)  "
    f"jitters {int(flagged.sum()):,}",
    [("kept", ~flagged, GREEN), ("jitters", flagged, AMBER)],
  ))

flag_video(static_uv, static_vis, rows, "Static jitters — one row per flicker")


In [ ]:
print(f"  static tol = {STATIC_DEPTH_TOL:g} m   mask_margin = {config.tracks.mask_margin} px   "
      f"match_radius = {config.tracks.match_radius:g} m")
print(f"  min_run_fraction = {STATIC_MIN_RUN_FRACTION:g}   flicker = {STATIC_FLICKER:g}")
print(f"\n  {len(static_view):,} candidates, vis on {100 * static_vis.mean():.1f}% of samples")
print(f"  gone {100 * static_gone.mean():.1f}%   jitters {100 * static_jitters.mean():.1f}%   "
      f"kept {100 * static_keep.mean():.1f}%")
